# 50 — Render the Biomedical Concept A-Box

Decision D4: **the BC layer only.** The Dataset Specialization layer is deferred —
it is where the volume is (13,922 variables, ~450,000 triples) and where the
remaining questions sit.

Output: `cosmos_bc_v1.instances.ttl`, canonicalized per decision D9.

Rendered directly with `rdflib` rather than through `linkml-convert` (decision
D8), so the published schema needs no constraint edits. `60_validate_instances`
runs the `linkml-convert` path on a sample as an independent check.

**What is not rendered:** the six concepts with no NCIt code, per decision D2.
`45_identity_probe.ipynb` lists them in `reports/unidentified_concepts.csv`.

Two authored shapes inside the core rendering, both named in `docs/decisions.md`
rather than left implicit: a category is a label-node, not a string (D18), and a
concept's use of a data element concept is a node of its own carrying `dataType`
and `exampleSet` (D21). Both mint IRIs under this repo's namespace — the first
time the core does so — and both are argued there.

## Configuration

In [ ]:
ROOT      = ".."
DOWNLOADS = "../downloads"
REPORTS   = "../reports"

BC_EXPORT = f"{DOWNLOADS}/cdisc_biomedical_concepts_latest.csv"
BC_TBOX   = f"{ROOT}/cosmos_bc_v1.ttl"
TARGET    = "cosmos_bc_v1.instances.ttl"

BC_NS   = "https://www.cdisc.org/cosmos/biomedical_concept_v1.0/"
OBO     = "http://purl.obolibrary.org/obo/NCIT_"
EVS     = "http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#"

VERSION = "0.3.0"
ONTOLOGY_IRI = "https://w3id.org/cdisc/cosmos/bc/"

# D18: a category token becomes a label-node here. D21: a concept's use of a
# data element concept becomes a node here. Both are this repo's IRIs; neither
# names anything CDISC named.
CATEGORY_NS = ONTOLOGY_IRI + "category/"
DEC_USE_NS  = ONTOLOGY_IRI

## The BC-level view

The flat export is at DEC-and-coding grain: 6,306 rows for 1,475 concepts, with
BC-level values repeated on every row. A concept's state is taken from its rows at
the **latest `package_date`** it appears in.

That rule is not asserted. Reconstructing `bc_hierarchy_full`, `bc_hierarchy_level`
and `dec_n` under it reproduced CDISC's own published hierarchy export exactly —
1,475 of 1,475 on all three, measured 2026-09-01. The publisher's view agrees with
the rule, which is why the hierarchy export is not fetched as an input.

In [ ]:
from pathlib import Path

import pandas as pd

export = pd.read_csv(BC_EXPORT, dtype=str, keep_default_na=False)

latest = export.package_date.groupby(export.bc_id).transform("max")
current = export[export.package_date == latest]

bc_level = current.drop_duplicates("bc_id").set_index("bc_id")

print(f"export rows          {len(export):>6,}")
print(f"rows at latest date  {len(current):>6,}")
print(f"biomedical concepts  {len(bc_level):>6,}")

## Which concepts are rendered

Decision D2: a concept's subject IRI is its NCIt code. A concept without one gets
no node, and this repo mints nothing for it.

In [ ]:
identified = bc_level[bc_level.ncit_code != ""]
unidentified = bc_level[bc_level.ncit_code == ""]

if not (unidentified.index.str.startswith("NEW_")).all():
    raise RuntimeError("a concept lacks an NCIt code without being a NEW_ placeholder")

print(f"rendered   {len(identified):>6,}")
print(f"omitted    {len(unidentified):>6,}  {sorted(unidentified.index)}")

## Render

Property IRIs come from the T-Box, so the A-Box and the ontology use one
vocabulary. Enum-ranged values resolve to the permissible-value IRIs the T-Box
declares, rather than being emitted as strings.

The identifier is not repeated as a property: it *is* the subject IRI. The bare
code is carried as `dcterms:identifier` so a consumer joining on CDISC's own
column still can — the same treatment decision D3 gives the DSS mnemonic.

`coding` is identified by `system` + `code` composed into an IRI —
`http://loinc.org/` + `64098-7`. That is derived from two published fields, not
invented: the schema documents `system` as "the URL of the code system", and the
composition resolves (`https://loinc.org/64098-7`, LOINC status Active, confirmed
in a browser 2026-09-01; `curl` returns 403, a bot block, which proves nothing
either way). All 98 distinct codes compose to syntactically valid absolute IRIs,
and the only system present is LOINC.

**No mapping predicate is asserted between a concept and its LOINC term.** The
coding node *is* that term; what it means relative to the biomedical concept is
left unsaid, because it is often not equivalence — the HCV RNA analysis in
`cdisc-for-ai` found 16 `narrowMatch` and one `broadMatch` with no `exactMatch`
anywhere. Asserting `skos:exactMatch` here would be exactly the unverified claim
this repo refuses to make.

Seven codes are referenced by more than one concept; identifying the coding by its
IRI merges those onto one node, which is correct — it is one LOINC term.

**A category is a label-node, not a string** (decision D18). The token is
published; the node at `…/cosmos/bc/category/{token}` carries it as `rdfs:label`
and asserts nothing else — not that it is a concept, not what it resolves to.
Resolving a token to the concept it names is an authored join and belongs to the
overlay. The slug is the token percent-encoded with nothing left unescaped, so it
is exact and reversible: 33 of the 408 tokens in the export carry a character
outside letters, digits and space (`Tumor/Lesion Results`, `Alzheimer's …`,
`RECIST 1.1`), and no two tokens differ only in case. 405 of the 408 land on a
rendered concept; `Surveys`, `Acceptability Surveys` and `Questionnaires` occur
only on the six unidentified concepts decision D2 omits. Synonyms stay literals —
measured, they are shared by almost nothing.

In [ ]:
from urllib.parse import quote

from rdflib import BNode, Graph, Literal, Namespace, URIRef
from rdflib.namespace import DCTERMS, OWL, RDF, RDFS, SKOS, XSD

COSMOS_BC = Namespace(BC_NS)
VANN = Namespace("http://purl.org/vocab/vann/")

tbox = Graph().parse(BC_TBOX, format="turtle")


def enum_value(enum_name, value):
    """Permissible-value IRI as declared in the T-Box; fail if it is not there."""
    iri = URIRef(f"{BC_NS}{enum_name}#{value}")
    if (iri, RDF.type, OWL.Class) not in tbox:
        raise RuntimeError(f"{value!r} is not a permissible value of {enum_name}")
    return iri


def split(value):
    return [part.strip() for part in value.split(";") if part.strip()]


def concept_iri(code):
    return URIRef(OBO + code)


g = Graph()

for bc_id, row in identified.iterrows():
    subject = concept_iri(row.ncit_code)

    g.add((subject, RDF.type, COSMOS_BC.BiomedicalConcept))
    g.add((subject, SKOS.exactMatch, URIRef(EVS + row.ncit_code)))
    g.add((subject, DCTERMS.identifier, Literal(bc_id)))

    g.add((subject, COSMOS_BC.shortName, Literal(row.short_name)))
    g.add((subject, COSMOS_BC.definition, Literal(row.definition)))
    g.add((subject, COSMOS_BC.ncitCode, Literal(row.ncit_code)))
    g.add((subject, COSMOS_BC.packageDate, Literal(row.package_date, datatype=XSD.date)))
    g.add((subject, COSMOS_BC.packageType, enum_value("PackageTypeEnum", "bc")))

    for category in split(row.bc_categories):
        node = URIRef(CATEGORY_NS + quote(category, safe=""))
        g.add((subject, COSMOS_BC.categories, node))
        g.add((node, RDFS.label, Literal(category)))
    for synonym in split(row.synonyms):
        g.add((subject, COSMOS_BC.synonyms, Literal(synonym)))
    for scale in split(row.result_scales):
        g.add((subject, COSMOS_BC.resultScales, enum_value("BiomedicalConceptResultScaleEnum", scale)))

    if row.parent_bc_id:
        parent = bc_level.loc[row.parent_bc_id] if row.parent_bc_id in bc_level.index else None
        if parent is not None and parent.ncit_code:
            g.add((subject, COSMOS_BC.parentConceptId, concept_iri(parent.ncit_code)))

print(f"{len(g):,} triples after concept headers")

## Codings and data element concepts

Both are inlined children of a concept in the published schema, and both are
repeated across the flat export's rows, so both are deduplicated.

**A data element concept is rendered twice, at two grains** (decision D21).
Measured at this pin, `dec_label` and `ncit_dec_code` are constant per DEC — 0 of
226 vary — but `data_type` varies for 8 DECs and `example_set` for 68: they are
properties of the **(concept, DEC) pair**, not of the DEC. `C70856` Observation
Result carries seven data types across the package. So the shared NCIt node keeps
what is constant — identity, `shortName`, `ncitCode` — and the concept's *use* of
it, a node at `…/cosmos/bc/{concept}/dec/{DEC}`, carries `dataType` and
`exampleSet`, linked to the shared node by `conceptId` rendered as an edge. This
is the shape `75_render_qbc.ipynb` already uses for the overlay, so core and
overlay attach a claim about the same pair at the same place.

The use-node is the object the schema actually describes: `dataElementConcepts`
inlines a `DataElementConcept` whose `conceptId` is its identifier. The shared
node is what that identifier resolves to under decision D2. Both are typed
`DataElementConcept`; the conformance report says what that costs.

The renderer asserts that no pair carries more than one `data_type` or
`example_set` — if it ever does, the pair is not the grain either.

In [ ]:
codings = 0
decs = set()
dec_uses = 0
dangling_dec = []

pair_grain = current[current.dec_id != ""].groupby(["bc_id", "dec_id"]).agg(
    data_types=("data_type", "nunique"), example_sets=("example_set", "nunique"))
if (pair_grain > 1).any().any():
    raise RuntimeError("a (concept, DEC) pair carries more than one data_type or example_set")

for bc_id, rows in current.groupby("bc_id"):
    if not rows.iloc[0].ncit_code:
        continue
    subject = concept_iri(rows.iloc[0].ncit_code)

    for _, row in rows.drop_duplicates(["code", "system"]).iterrows():
        if not row.code:
            continue
        coding = URIRef(row.system + row.code)
        g.add((subject, COSMOS_BC.coding, coding))
        g.add((coding, RDF.type, COSMOS_BC.Coding))
        g.add((coding, COSMOS_BC.code, Literal(row.code)))
        g.add((coding, COSMOS_BC.system, Literal(row.system)))
        if row.system_name:
            g.add((coding, COSMOS_BC.systemName, Literal(row.system_name)))
        codings += 1

    for _, row in rows.drop_duplicates("dec_id").iterrows():
        if not row.dec_id:
            continue
        if not row.ncit_dec_code:
            dangling_dec.append((bc_id, row.dec_id))
            continue
        dec = concept_iri(row.ncit_dec_code)
        use = URIRef(f"{DEC_USE_NS}{rows.iloc[0].ncit_code}/dec/{row.ncit_dec_code}")
        dec_uses += 1

        # D21: the concept's use of the DEC carries what varies by pair.
        g.add((subject, COSMOS_BC.dataElementConcepts, use))
        g.add((use, RDF.type, COSMOS_BC.DataElementConcept))
        g.add((use, COSMOS_BC.conceptId, dec))
        g.add((use, COSMOS_BC.shortName, Literal(row.dec_label)))
        if row.data_type:
            g.add((use, COSMOS_BC.dataType, enum_value("DataElementConceptDataTypeEnum", row.data_type)))
        for example in split(row.example_set):
            g.add((use, COSMOS_BC.exampleSet, Literal(example)))

        if row.dec_id in decs:
            continue
        decs.add(row.dec_id)
        # The shared node keeps what is constant per DEC: identity and label.
        g.add((dec, RDF.type, COSMOS_BC.DataElementConcept))
        g.add((dec, SKOS.exactMatch, URIRef(EVS + row.ncit_dec_code)))
        g.add((dec, DCTERMS.identifier, Literal(row.dec_id)))
        g.add((dec, COSMOS_BC.shortName, Literal(row.dec_label)))
        g.add((dec, COSMOS_BC.ncitCode, Literal(row.ncit_dec_code)))

print(f"codings rendered              {codings:>6,}")
print(f"data element concepts         {len(decs):>6,}")
print(f"(concept, DEC) uses           {dec_uses:>6,}")
print(f"DEC references omitted (no NCIt code) {len(dangling_dec):>3,}  {sorted({d for _, d in dangling_dec})}")
print(f"{len(g):,} triples total")

## Concepts used at two layers

Nineteen NCIt codes are used as both a Biomedical Concept and a Data Element
Concept, so under decision D2 they resolve to one node carrying both types.

For the three Demographics cases — Race `C17049`, Sex `C28421`, Ethnic Group
`C16564` — that is correct rather than a compromise: each is a BC with one DSS in
DM and a DEC on one DM variable, so the concept is the whole content of the
observation and there is nothing for the two roles to be distinct from.

The rest are a mixed population and are **reported, not modelled around** — the
same treatment the unidentified concepts get. Fifteen have no DSS of their own as
a BC; seven appear in no DSS in either role. Whether a BC should exist that
nothing specializes is a curation question for CDISC, not something this rendering
should resolve.

In [ ]:
import csv

dss_export = pd.read_csv(f"{DOWNLOADS}/cdisc_sdtm_dataset_specializations_latest.csv",
                         dtype=str, keep_default_na=False)

bc_codes = set(bc_level[bc_level.ncit_code != ""].ncit_code)
dec_codes = set(current[current.ncit_dec_code != ""].ncit_dec_code)

dual = []
for code in sorted(bc_codes & dec_codes):
    as_bc = dss_export[dss_export.bc_id == code]
    as_dec = dss_export[dss_export.dec_id == code]
    bc_row = current[current.ncit_code == code].iloc[0]
    dec_row = current[current.ncit_dec_code == code].iloc[0]
    dual.append({
        "ncit_code": code,
        "bc_short_name": bc_row.short_name,
        "dec_label": dec_row.dec_label,
        "labels_agree": bc_row.short_name == dec_row.dec_label,
        "dss_as_bc": as_bc.vlm_group_id.nunique(),
        "domains_as_bc": ";".join(sorted(as_bc.domain.unique())),
        "rows_as_dec": len(as_dec),
        "domains_as_dec": ";".join(sorted(as_dec.domain.unique())),
        "variables_as_dec": ";".join(sorted(as_dec.sdtm_variable.unique())),
    })

report = Path(REPORTS, "dual_role_concepts.csv")
with open(report, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(dual[0]))
    writer.writeheader()
    writer.writerows(dual)

print(f"wrote {report}: {len(dual)} concepts used at both layers")
print(f"   labels disagree: {sum(1 for d in dual if not d['labels_agree'])}"
      f"  {[d['ncit_code'] for d in dual if not d['labels_agree']]}")
print(f"   with a DSS as a BC: {sum(1 for d in dual if d['dss_as_bc'])}")
print(f"   in no DSS at all:   {sum(1 for d in dual if not d['dss_as_bc'] and not d['rows_as_dec'])}")

## Ontology header and canonical write

The instance graph carries its own header, `owl:imports`-ing the T-Box so a
consumer loading one gets the other. Same version and provenance discipline as
the T-Box (decisions D7 and D9).

In [ ]:
import json

from rdflib.compare import isomorphic, to_canonical_graph

INSTANCES_IRI = ONTOLOGY_IRI + "instances/"

meta = json.loads(Path(DOWNLOADS, ".fetch_meta_bc_export.json").read_text(encoding="utf-8"))

ontology = URIRef(INSTANCES_IRI)
g.add((ontology, RDF.type, OWL.Ontology))
g.add((ontology, OWL.imports, URIRef(ONTOLOGY_IRI)))
g.add((ontology, RDFS.label, Literal("CDISC COSMoS Biomedical Concepts (instances)")))
g.add((ontology, RDFS.comment, Literal(
    "Biomedical Concept instances rendered from the pinned COSMoS export. "
    "Concepts with no NCIt code are not represented. Draft - not a normative CDISC artifact.")))
g.add((ontology, DCTERMS.source, URIRef(meta["raw_url"])))
g.add((ontology, DCTERMS.identifier, Literal(meta["package_date"])))
g.add((ontology, OWL.versionIRI, URIRef(INSTANCES_IRI + VERSION)))
g.add((ontology, OWL.versionInfo, Literal(f"v{VERSION}")))

canonical = to_canonical_graph(g)
if not isomorphic(canonical, g) or len(canonical) != len(g):
    raise RuntimeError("canonicalization changed the graph")

out = Graph()
for triple in canonical:
    out.add(triple)
out.bind("cosmos_bc", BC_NS)
out.bind("dcterms", DCTERMS)
out.bind("skos", SKOS)
out.bind("NCIT", OBO)

turtle = out.serialize(format="turtle")
Path(ROOT, TARGET).write_text(turtle, encoding="utf-8")
print(f"{TARGET}  {len(out):,} triples  {len(turtle):,} chars")

## Confirm

In [ ]:
concepts = set(g.subjects(RDF.type, COSMOS_BC.BiomedicalConcept))
dec_nodes = set(g.subjects(RDF.type, COSMOS_BC.DataElementConcept))
parents = set(g.subject_objects(COSMOS_BC.parentConceptId))

use_nodes = {s for s in dec_nodes if str(s).startswith(DEC_USE_NS)}
shared_nodes = dec_nodes - use_nodes
categories = set(g.objects(None, COSMOS_BC.categories))

print(f"BiomedicalConcept nodes   {len(concepts):>6,}   (expected {len(identified):,})")
print(f"DataElementConcept nodes  {len(dec_nodes):>6,}   shared {len(shared_nodes):,} + uses {len(use_nodes):,} (D21)")
print(f"category nodes            {len(categories):>6,}   (D18)")
print(f"parent edges              {len(parents):>6,}")
print(f"coding nodes              {len(set(g.objects(None, COSMOS_BC.coding))):>6,}")
print(f"blank nodes               {len({s for s in g.subjects() if isinstance(s, BNode)}):>6,}")

if len(concepts) != len(identified):
    raise RuntimeError("concept count does not match the rendered set")

dangling = {o for _, o in parents} - concepts
print(f"parent edges with no target {len(dangling):>4,}")

if len(use_nodes) != dec_uses:
    raise RuntimeError("use-node count does not match the rendered pairs")
if any((c, None, None) not in g for c in categories):
    raise RuntimeError("a category edge points at a node with no label")
if any((s, COSMOS_BC.dataType, None) in g for s in shared_nodes):
    raise RuntimeError("D21: dataType asserted on a shared DEC node")
if any(str(c) != CATEGORY_NS + quote(str(g.value(c, RDFS.label)), safe="") for c in categories):
    raise RuntimeError("D18: a category IRI does not round-trip from its label")